# Setup & Import Library

In [14]:
import pandas as pd
import numpy as np
from prophet import Prophet
import plotly.graph_objs as go
from sklearn.metrics import mean_absolute_error, mean_squared_error
from joblib import Parallel, delayed
import logging
import warnings

# === SETUP ===
warnings.filterwarnings("ignore")
logging.getLogger('prophet').setLevel(logging.ERROR)
logging.getLogger('cmdstanpy').disabled = True

# Target tahun untuk Pivot dan Filter CSV
TARGET_TAHUN = 2025

# Data Preprocessing Pipeline

In [15]:
def load_and_preprocess_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Membersihkan dan memformat DataFrame mentah untuk pemodelan Prophet.
    
    Proses yang dilakukan meliputi:
    1. Penghapusan duplikasi dan nilai kosong pada kolom esensial.
    2. Konversi format string angka (mengandung koma) menjadi float.
    3. Penyaringan nilai volume negatif.
    4. Pembuatan kolom tanggal ('ds') dan target ('y') sesuai standar Prophet.
    5. Pengurutan data historis berdasarkan tanggal dan spesialisasi.

    Parameters:
        df (pd.DataFrame): DataFrame mentah hasil ekstraksi.

    Returns:
        pd.DataFrame: DataFrame bersih yang siap digunakan untuk training.
    """
    data = df.copy()
    print("Memulai proses pembersihan data...")
    
    data.drop_duplicates(inplace=True)
    data.dropna(subset=['year', 'month', 'volume'], inplace=True)
    
    if data['volume'].dtype in ['object', 'O']:
        data['volume'] = data['volume'].astype(str).str.replace(',', '.').astype(float)
        
    if 'revenue' in data.columns and data['revenue'].dtype in ['object', 'O']:
        data['revenue'] = data['revenue'].astype(str).str.replace(',', '.').astype(float)
        
    data = data[data['volume'] >= 0]
    data['year'] = data['year'].astype(int)
    data['month'] = data['month'].astype(int)
    data['ds'] = pd.to_datetime(data[['year', 'month']].assign(DAY=1))
    
    data.rename(columns={'volume': 'y'}, inplace=True)
    
    data_actual = (
        data[data['is_forecast'] == 0].copy()
        if 'is_forecast' in data.columns
        else data.copy()
    )
    
    data_actual['monitor_month'] = pd.to_datetime(data_actual['monitor_month']).dt.strftime('%Y-%m-%d')
    
    if 'specialism_code' in data_actual.columns:
        data_actual.sort_values(by=['specialism_code', 'ds'], inplace=True)
    else:
        data_actual.sort_values(by=['ds'], inplace=True)
        
    data_actual.reset_index(drop=True, inplace=True)
    print(f"✅ Preprocessing Selesai! Data bersih siap digunakan: {len(data_actual)} baris.")
    
    return data_actual

# Helper Functions (Statistik & Thresholding)

In [16]:
def winsorize_series(df: pd.DataFrame, col: str = 'y', lower: float = 0.02, upper: float = 0.98) -> pd.DataFrame:
    """
    Membatasi nilai ekstrem (outliers) pada deret waktu menggunakan metode Winsorization.

    Parameters:
        df (pd.DataFrame): DataFrame input.
        col (str): Nama kolom yang akan di-winsorize.
        lower (float): Persentil batas bawah (default 2%).
        upper (float): Persentil batas atas (default 98%).

    Returns:
        pd.DataFrame: DataFrame dengan nilai ekstrem yang telah dibatasi.
    """
    if len(df) < 10:
        return df
    q_low = df[col].quantile(lower)
    q_high = df[col].quantile(upper)
    df = df.copy()
    df[col] = df[col].clip(lower=q_low, upper=q_high)
    return df

def auto_changepoint_scale(df: pd.DataFrame) -> float:
    """
    Menentukan nilai changepoint_prior_scale Prophet secara dinamis 
    berdasarkan koefisien variasi (Coefficient of Variation) data.

    Parameters:
        df (pd.DataFrame): DataFrame dengan kolom target 'y'.

    Returns:
        float: Nilai scale yang direkomendasikan.
    """
    mean_val = df['y'].mean()
    if mean_val <= 0:
        return 0.05
    cov = df['y'].std() / mean_val
    if cov < 0.15:   return 0.01
    elif cov < 0.35: return 0.05
    elif cov < 0.60: return 0.15
    else:            return 0.30

def clip_forecast_negative(forecast: pd.DataFrame) -> pd.DataFrame:
    """
    Mencegah hasil prediksi Prophet memiliki nilai negatif.

    Parameters:
        forecast (pd.DataFrame): DataFrame hasil prediksi model Prophet.

    Returns:
        pd.DataFrame: DataFrame prediksi yang dinormalisasi ke minimum 0.
    """
    cols = [c for c in ['yhat', 'yhat_lower', 'yhat_upper'] if c in forecast.columns]
    forecast = forecast.copy()
    forecast[cols] = forecast[cols].clip(lower=0)
    return forecast

def is_sparse_series(df: pd.DataFrame, col: str = 'y', zero_threshold: float = 0.50) -> bool:
    """
    Mendeteksi apakah deret waktu terlalu banyak memiliki nilai nol (sparse).

    Parameters:
        df (pd.DataFrame): DataFrame deret waktu.
        col (str): Nama kolom target.
        zero_threshold (float): Ambang batas persentase nilai nol.

    Returns:
        bool: True jika data sparse, False jika memadai untuk prediksi.
    """
    if len(df) == 0:
        return True
    zero_frac = (df[col] == 0).sum() / len(df)
    return zero_frac > zero_threshold

def auto_yearly_fourier(n_points: int) -> int:
    """
    Menyesuaikan orde Fourier untuk musiman tahunan berdasarkan jumlah titik data historis.

    Parameters:
        n_points (int): Jumlah observasi historis (bulan).

    Returns:
        int: Orde Fourier yang direkomendasikan.
    """
    if n_points < 24:
        return 2
    elif n_points < 36:
        return 4
    else:
        return 6

# Core Prophet Model Training

In [17]:
def train_evaluate_predict_v5(df_specialty: pd.DataFrame, test_months: int = 6, periods_to_forecast: int = 12) -> dict:
    """
    Melatih model Prophet, mengevaluasi akurasi menggunakan data uji terakhir (backtesting), 
    lalu memprediksi periode ke depan.

    Parameters:
        df_specialty (pd.DataFrame): Data historis spesifik untuk satu entitas/spesialisasi.
        test_months (int): Jumlah bulan terakhir yang dialokasikan untuk evaluasi model.
        periods_to_forecast (int): Jumlah bulan ke depan untuk diprediksi dari data terakhir.

    Returns:
        dict: Kamus berisi status (skipped), metrik error, objek model, hasil prediksi (forecast), 
              data agregat asli (df_agg), dan nilai changepoint scale.
    """
    df_agg = df_specialty.groupby('ds')['y'].sum(min_count=1).reset_index()
    df_agg.dropna(subset=['y'], inplace=True)
    
    result = {
        'metrics': {'MAE': None, 'RMSE': None, 'WMAPE': None},
        'model': None,
        'forecast': None,
        'df_agg': df_agg,
        'changepoint_scale': None,
        'skipped': False,
        'skip_reason': None,
    }
    
    # 1. Sparse Check
    if is_sparse_series(df_agg, col='y', zero_threshold=0.50):
        result['skipped'] = True
        result['skip_reason'] = 'sparse (>50% nol)'
        return result
        
    # 2. Winsorization & Dinamis Konfigurasi
    df_agg = winsorize_series(df_agg, col='y', lower=0.02, upper=0.98)
    result['df_agg'] = df_agg
    n_points = len(df_agg)
    cps = auto_changepoint_scale(df_agg)
    yearly_f = auto_yearly_fourier(n_points)
    result['changepoint_scale'] = cps

    def build_model():
        m = Prophet(
            changepoint_prior_scale=cps,
            seasonality_prior_scale=1.0,
            yearly_seasonality=yearly_f,
            interval_width=0.80,
            weekly_seasonality=False,
            daily_seasonality=False,
            mcmc_samples=0,
        )
        m.add_country_holidays(country_name='NL')
        if n_points >= 24:
            m.add_seasonality(name='quarterly', period=91.25, fourier_order=4)
        return m

    # 3. Skema Jika Data Terlalu Sedikit
    min_required = test_months + 6
    if n_points <= min_required:
        final_model = build_model()
        final_model.fit(df_agg)
        future = final_model.make_future_dataframe(periods=periods_to_forecast, freq='MS')
        forecast = clip_forecast_negative(final_model.predict(future))
        result['forecast'] = forecast
        result['model'] = final_model
        return result

    # 4. Evaluasi (Backtesting)
    train_df = df_agg.iloc[:-test_months].copy()
    test_df = df_agg.iloc[-test_months:].copy()
    
    eval_model = build_model()
    eval_model.fit(train_df)
    total_periods = test_months + periods_to_forecast
    future_eval = eval_model.make_future_dataframe(periods=total_periods, freq='MS')
    fc_eval = clip_forecast_negative(eval_model.predict(future_eval))
    
    fc_test = fc_eval[fc_eval['ds'].isin(test_df['ds'])]['yhat'].values
    y_true = test_df['y'].values
    n_matched = min(len(y_true), len(fc_test))
    
    if n_matched > 0:
        y_true_m = y_true[:n_matched]
        fc_test_m = fc_test[:n_matched]
        sum_err = np.sum(np.abs(y_true_m - fc_test_m))
        sum_act = np.sum(np.abs(y_true_m))
        result['metrics']['MAE'] = mean_absolute_error(y_true_m, fc_test_m)
        result['metrics']['RMSE'] = np.sqrt(mean_squared_error(y_true_m, fc_test_m))
        result['metrics']['WMAPE'] = (sum_err / sum_act) * 100 if sum_act != 0 else 0
        
    # 5. Final Model Training (Keseluruhan Data)
    final_model = build_model()
    final_model.fit(df_agg)
    future = final_model.make_future_dataframe(periods=periods_to_forecast, freq='MS')
    forecast = clip_forecast_negative(final_model.predict(future))
    
    result['forecast'] = forecast
    result['model'] = final_model
    return result

# Batch & Parallel Processing

In [18]:
def _worker(spec: str, mon: str, df_clean: pd.DataFrame, test_months: int) -> tuple:
    """
    Fungsi worker tunggal untuk diproses secara paralel oleh Joblib.
    """
    mon_date = pd.to_datetime(mon)
    df_train = df_clean[
        (df_clean['specialism_code'] == spec) &
        (df_clean['monitor_month'] == mon)
    ].copy()
    
    df_train = df_train[~(
        (df_train['year'] == mon_date.year) &
        (df_train['month'] == mon_date.month) &
        (df_train['is_forecast'] == 0)
    )]
    
    if len(df_train['ds'].unique()) < 2:
        return (spec, mon, None)
        
    hasil = train_evaluate_predict_v5(df_train, test_months=test_months)
    return (spec, mon, hasil)


def run_batch(df_clean: pd.DataFrame, specialties: list, q_monitors: list, test_months: int = 6, n_jobs: int = -1) -> dict:
    """
    Mengeksekusi proses pelatihan model secara paralel untuk setiap kombinasi spesialisasi dan bulan monitor.

    Parameters:
        df_clean (pd.DataFrame): DataFrame yang sudah dibersihkan.
        specialties (list): Daftar kode spesialisasi unik.
        q_monitors (list): Daftar tanggal monitor untuk diproses.
        test_months (int): Jumlah bulan alokasi test backtesting.
        n_jobs (int): Jumlah CPU core yang digunakan (-1 untuk semua core tersedia).

    Returns:
        dict: Struktur kamus tersarang (nested) berisi hasil pemodelan [spesialisasi][monitor_month].
    """
    tasks = [
        (spec, mon)
        for spec in specialties
        for mon in q_monitors
    ]
    
    print(f"🚀 Memulai batch training: {len(tasks)} kombinasi spec×snapshot")
    print(f"   Paralel dengan n_jobs={n_jobs} (joblib)\n")
    
    results = Parallel(n_jobs=n_jobs, backend='loky', verbose=5)(
        delayed(_worker)(spec, mon, df_clean, test_months)
        for spec, mon in tasks
    )
    
    forecasts_dict: dict = {}
    for spec, mon, hasil in results:
        if spec not in forecasts_dict:
            forecasts_dict[spec] = {}
        if hasil is not None:
            forecasts_dict[spec][mon] = hasil
            
    return forecasts_dict

# Read Data & Batch Execution

In [19]:
try:
    df_raw = pd.read_csv("data/raw/Data 1-2.csv", sep=";", decimal=",")
    df_raw.dropna(subset=['monitor_month'], inplace=True)
    df_raw['monitor_month'] = pd.to_datetime(df_raw['monitor_month']).dt.strftime('%Y-%m-%d')
    df_clean = load_and_preprocess_data(df_raw)
except Exception as e:
    print(f"❌ Terjadi kesalahan saat membaca data: {e}")
    raise

# Persiapan variabel referensi
specialties = df_clean['specialism_code'].unique().tolist()
all_mons = df_raw['monitor_month'].dropna().astype(str).unique()
q_monitors = sorted([
    m for m in all_mons
    if '-' in m and int(m.split('-')[1]) in [1, 4, 7, 10]
])
TEST_MONTHS = 6

# ==========================================
# EKSEKUSI BATCH PARALEL
# ==========================================
forecasts_dict = run_batch(
    df_clean=df_clean,
    specialties=specialties,
    q_monitors=q_monitors,
    test_months=TEST_MONTHS,
    n_jobs=-1
)

Memulai proses pembersihan data...
✅ Preprocessing Selesai! Data bersih siap digunakan: 525886 baris.
🚀 Memulai batch training: 10 kombinasi spec×snapshot
   Paralel dengan n_jobs=-1 (joblib)



[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of  10 | elapsed:    2.1s remaining:    8.6s
[Parallel(n_jobs=-1)]: Done   5 out of  10 | elapsed:    2.5s remaining:    2.5s
[Parallel(n_jobs=-1)]: Done   8 out of  10 | elapsed:    3.2s remaining:    0.7s
[Parallel(n_jobs=-1)]: Done  10 out of  10 | elapsed:    3.6s finished


# Laporan Akurasi

In [20]:
print("\n" + "=" * 75)
print(" 📊 LAPORAN AKURASI PREDIKSI AI (Backtesting 6 Bulan Terakhir)")
print("=" * 75)

wmape_all = []

for spec, mon_dict in forecasts_dict.items():
    print(f"Spesialisasi: {spec}")
    for mon, hasil in mon_dict.items():
        if hasil.get('skipped'):
            print(f"  > Snapshot [{mon}] | ⚠️  Dilewati — {hasil['skip_reason']}")
            continue
            
        metrics = hasil['metrics']
        cps = hasil.get('changepoint_scale', '-')
        
        if metrics['WMAPE'] is not None:
            wmape_all.append(metrics['WMAPE'])
            print(
                f"  > Snapshot [{mon}] | "
                f"WMAPE: {metrics['WMAPE']:>5.2f}% | "
                f"MAE: {metrics['MAE']:>6.2f} | "
                f"RMSE: {metrics['RMSE']:>6.2f} | "
                f"CPS: {cps}"
            )
        else:
            print(f"  > Snapshot [{mon}] | Data historis belum cukup untuk mengukur akurasi.")
    print("-" * 75)

if wmape_all:
    print(f"\n📈 Rata-rata WMAPE keseluruhan : {np.mean(wmape_all):.2f}%")
    print(f"   Median WMAPE                : {np.median(wmape_all):.2f}%")
    print(f"   WMAPE terbaik               : {np.min(wmape_all):.2f}%")
    print(f"   WMAPE terburuk              : {np.max(wmape_all):.2f}%")
    
print("\n✅ Proses batch training selesai!\n")


 📊 LAPORAN AKURASI PREDIKSI AI (Backtesting 6 Bulan Terakhir)
Spesialisasi: AGB:0303
  > Snapshot [2025-04-01] | WMAPE: 12.12% | MAE: 378.07 | RMSE: 425.68 | CPS: 0.01
  > Snapshot [2025-07-01] | WMAPE:  6.27% | MAE: 201.08 | RMSE: 231.69 | CPS: 0.01
  > Snapshot [2025-10-01] | WMAPE:  5.40% | MAE: 173.16 | RMSE: 187.23 | CPS: 0.01
  > Snapshot [2026-01-01] | WMAPE:  3.47% | MAE: 110.35 | RMSE: 156.21 | CPS: 0.01
  > Snapshot [2026-04-01] | WMAPE:  3.95% | MAE: 125.19 | RMSE: 161.58 | CPS: 0.01
---------------------------------------------------------------------------
Spesialisasi: AGB:0305
  > Snapshot [2025-04-01] | WMAPE: 11.93% | MAE: 156.97 | RMSE: 180.29 | CPS: 0.01
  > Snapshot [2025-07-01] | WMAPE:  9.23% | MAE: 120.27 | RMSE: 137.33 | CPS: 0.01
  > Snapshot [2025-10-01] | WMAPE:  3.78% | MAE:  45.14 | RMSE:  52.38 | CPS: 0.01
  > Snapshot [2026-01-01] | WMAPE:  7.50% | MAE:  91.33 | RMSE: 109.55 | CPS: 0.01
  > Snapshot [2026-04-01] | WMAPE:  7.26% | MAE:  88.86 | RMSE: 102.

# Visualisasi Interaktif (Plotly)

In [21]:
def plot_interactive_forecast(hasil_dict: dict, specialty_name: str, snapshot_date: str):
    """
    Membuat visualisasi grafik time series interaktif untuk observasi aktual vs prediksi Prophet.

    Parameters:
        hasil_dict (dict): Dictionary hasil prediksi fungsi utama.
        specialty_name (str): Nama atau kode spesialisasi.
        snapshot_date (str): Tanggal eksekusi snapshot (monitor_month).

    Returns:
        go.Figure: Objek figure dari Plotly siap untuk dirender.
    """
    df_agg = hasil_dict['df_agg']
    forecast = hasil_dict['forecast']
    metrics = hasil_dict['metrics']
    cps = hasil_dict.get('changepoint_scale', '-')
    
    fig = go.Figure()
    
    # Trace Confidence Interval
    fig.add_trace(go.Scatter(
        x=forecast['ds'].tolist() + forecast['ds'].tolist()[::-1],
        y=forecast['yhat_upper'].tolist() + forecast['yhat_lower'].tolist()[::-1],
        fill='toself', fillcolor='rgba(0, 176, 246, 0.2)',
        line=dict(color='rgba(255,255,255,0)'), hoverinfo="skip",
        showlegend=True, name='80% Confidence Interval'
    ))
    
    # Trace Batas Atas
    fig.add_trace(go.Scatter(
        x=forecast['ds'], y=forecast['yhat_upper'], mode='lines',
        line=dict(color='rgba(0, 176, 246, 1)', width=1),
        showlegend=False, name='Batas Atas', hovertemplate='%{y}<extra></extra>'
    ))
    
    # Trace Garis Prediksi Utama (Forecast)
    fig.add_trace(go.Scatter(
        x=forecast['ds'], y=forecast['yhat'], mode='lines',
        line=dict(color='blue', width=2.5),
        name='Forecasted Volume', hovertemplate='%{y}<extra></extra>'
    ))
    
    # Trace Batas Bawah
    fig.add_trace(go.Scatter(
        x=forecast['ds'], y=forecast['yhat_lower'], mode='lines',
        line=dict(color='rgba(0, 176, 246, 1)', width=1),
        showlegend=False, name='Batas Bawah', hovertemplate='%{y}<extra></extra>'
    ))
    
    # Trace Nilai Aktual
    fig.add_trace(go.Scatter(
        x=df_agg['ds'], y=df_agg['y'], mode='markers+lines',
        line=dict(color='black', dash='dot', width=2),
        marker=dict(color='black', size=6, symbol='circle'),
        name='Actual Volume', hovertemplate='%{y}<extra></extra>'
    ))
    
    # Garis Pembatas Vertikal (Titik potong actual ke forecast)
    last_actual_date = df_agg['ds'].max()
    fig.add_vline(x=last_actual_date, line_width=2, line_dash="dash", line_color="red")
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='lines',
        line=dict(color='red', dash='dash', width=2), name='Start of Forecast'
    ))
    
    wmape_text = f"{metrics['WMAPE']:.2f}%" if metrics['WMAPE'] is not None else "N/A"
    mae_text = f"{metrics['MAE']:.2f}" if metrics['MAE'] is not None else "N/A"
    
    fig.update_layout(
        title=(
            f'<b>Market Agnostic Forecast: Specialty {specialty_name}</b>'
            f'<br><sup>Snapshot Terbaru ({snapshot_date}) | '
            f'WMAPE = {wmape_text} | MAE = {mae_text} | CPS = {cps}</sup>'
        ),
        xaxis_title='Timeline (Bulan)', yaxis_title='Volume Produksi',
        template='plotly_white', hovermode='x unified',
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1, font=dict(size=10), itemwidth=30),
        margin=dict(t=80)
    )
    return fig

# Tampilkan Visualisasi hanya untuk data terbaru
print("Menghasilkan visualisasi interaktif final (Hanya Snapshot Terbaru)...\n")
for spec, mon_dict in forecasts_dict.items():
    if mon_dict:
        latest_mon = sorted(list(mon_dict.keys()))[-1]
        hasil_terbaru = mon_dict[latest_mon]
        if (not hasil_terbaru.get('skipped')) and hasil_terbaru['forecast'] is not None:
            fig = plot_interactive_forecast(hasil_terbaru, spec, latest_mon)
            fig.show()

Menghasilkan visualisasi interaktif final (Hanya Snapshot Terbaru)...



# Konstruksi Master Pivot Prophet

In [22]:
print("⚙️ Menyiapkan data Prophet untuk Pivot Table...\n")
prophet_rows = []

for spec, mon_dict in forecasts_dict.items():
    for mon, hasil in mon_dict.items():
        if hasil.get('skipped') or hasil['forecast'] is None:
            continue
            
        fc = hasil['forecast'][['ds', 'yhat']].copy()
        
        # Prophet mengekspor seluruh 'yhat' (prediksi) untuk perbandingan periode yang sama
        fc['monitor_month'] = mon
        fc['specialism_code'] = spec
        prophet_rows.append(fc)

if prophet_rows:
    df_prophet_master = pd.concat(prophet_rows, ignore_index=True)
    df_prophet_master['year'] = df_prophet_master['ds'].dt.year
    df_prophet_master['month'] = df_prophet_master['ds'].dt.month
else:
    df_prophet_master = pd.DataFrame()

⚙️ Menyiapkan data Prophet untuk Pivot Table...



# Pivot Table Engine (Terminal & Pandas Dataframe)

In [23]:
def print_text_pivot(df_mentah: pd.DataFrame, df_pro_master: pd.DataFrame, target_year: int, target_spec: str):
    """
    Mencetak pivot table volume (Aktual, SQL, Prophet) secara tekstual ke konsol/terminal.

    Parameters:
        df_mentah (pd.DataFrame): DataFrame berisi data historis mentah.
        df_pro_master (pd.DataFrame): Master DataFrame hasil prediksi Prophet.
        target_year (int): Tahun yang difilter.
        target_spec (str): Kode spesialisasi yang difilter.
    """
    df_yr = df_mentah[
        (df_mentah['year'] == target_year) &
        (df_mentah['specialism_code'] == target_spec)
    ].copy()
    
    if df_yr.empty:
        return
        
    df_yr.drop_duplicates(inplace=True)
    df_yr.dropna(subset=['year', 'month', 'volume'], inplace=True)
    
    if df_yr['volume'].dtype in ['object', 'O']:
        df_yr['volume'] = df_yr['volume'].astype(str).str.replace(',', '.').astype(float)
        
    sql_grp = (df_yr.groupby(['monitor_month', 'is_forecast', 'month'])['volume']
               .sum().unstack('month'))
    if not sql_grp.empty:
        sql_grp.columns = [int(c) for c in sql_grp.columns]

    pro_grp = pd.DataFrame()
    if not df_pro_master.empty:
        pro_yr = df_pro_master[
            (df_pro_master['year'] == target_year) &
            (df_pro_master['specialism_code'] == target_spec)
        ]
        if not pro_yr.empty:
            pro_grp = pro_yr.groupby(['monitor_month', 'month'])['yhat'].sum().unstack('month')
            pro_grp.columns = [int(c) for c in pro_grp.columns]

    monitors_in_year = sorted(df_yr['monitor_month'].unique())
    q_monitors_yr = [m for m in monitors_in_year if int(m.split('-')[1]) in [1, 4, 7, 10]]
    months = list(range(1, 13))
    sep = "-" * 140
    col_hdr = f"{'Monitor / Tipe':<30}" + "".join(f"{str(m):>9}" for m in months) + f"{'Grand Total':>13}"

    def fmt_num(v, width):
        s = f"{round(v):,}".replace(",", ".")
        return s.rjust(width)

    def fmt_row(label, data, months):
        vals = []
        for m in months:
            v = data.get(m, np.nan)
            vals.append(fmt_num(v, 9) if pd.notna(v) else "         ")
        total = sum(data.get(m, 0) for m in months if pd.notna(data.get(m, np.nan)))
        return f"{label:<30}" + "".join(vals) + fmt_num(total, 13)

    print("\n" + "=" * 140)
    print(f"  PIVOT TABLE — Spesialisasi: {target_spec}  |  Filter: Tahun = {target_year}")
    print("=" * 140)
    print(col_hdr)
    print(sep)

    for mon in q_monitors_yr:
        row_0 = sql_grp.loc[(mon, 0)].to_dict() if (mon, 0) in sql_grp.index else {}
        row_1 = sql_grp.loc[(mon, 1)].to_dict() if (mon, 1) in sql_grp.index else {}
        row_p = pro_grp.loc[mon].to_dict() if mon in pro_grp.index else {}

        if pd.notna(row_0.get(12, np.nan)):
            row_1 = {}
            row_p = {}

        if row_1:
            row_p = {m: v for m, v in row_p.items() if pd.notna(row_1.get(m, np.nan))}
        else:
            row_p = {}

        mon_data = {}
        for m in months:
            if pd.notna(row_0.get(m, np.nan)): mon_data[m] = row_0[m]
            elif pd.notna(row_1.get(m, np.nan)): mon_data[m] = row_1[m]

        print(fmt_row(f"■ {mon}", mon_data, months))
        if row_0: print(fmt_row("    0  Aktual", row_0, months))
        if row_1: print(fmt_row("    1  Forecast SQL", row_1, months))
        if row_p: print(fmt_row("    P  Forecast Prophet", row_p, months))
        print(sep)

    gt_all = {}
    for m in months:
        gt_all[m] = sql_grp[m].sum() if m in sql_grp.columns else np.nan
    print(fmt_row("GRAND TOTAL", gt_all, months))
    print("=" * 140)


def build_pivot_dataframe(df_mentah: pd.DataFrame, df_pro_master: pd.DataFrame, target_year: int, target_spec: str) -> pd.DataFrame:
    """
    Membangun struktur DataFrame Pivot untuk keperluan ekspor ke file Excel.

    Parameters:
        df_mentah (pd.DataFrame): DataFrame historis mentah.
        df_pro_master (pd.DataFrame): Master DataFrame Prophet forecast.
        target_year (int): Tahun yang difilter.
        target_spec (str): Spesialisasi yang diproses.

    Returns:
        pd.DataFrame: DataFrame yang memuat representasi tabel pivot per bulan (Jan - Des).
    """
    df_yr = df_mentah[
        (df_mentah['year'] == target_year) &
        (df_mentah['specialism_code'] == target_spec)
    ].copy()
    
    if df_yr.empty: return pd.DataFrame()

    df_yr.drop_duplicates(inplace=True)
    df_yr.dropna(subset=['year', 'month', 'volume'], inplace=True)
    
    if df_yr['volume'].dtype in ['object', 'O']:
        df_yr['volume'] = df_yr['volume'].astype(str).str.replace(',', '.').astype(float)

    sql_grp = (df_yr.groupby(['monitor_month', 'is_forecast', 'month'])['volume']
               .sum().unstack('month'))
    if not sql_grp.empty: sql_grp.columns = [int(c) for c in sql_grp.columns]

    pro_grp = pd.DataFrame()
    if not df_pro_master.empty:
        pro_yr = df_pro_master[
            (df_pro_master['year'] == target_year) &
            (df_pro_master['specialism_code'] == target_spec)
        ]
        if not pro_yr.empty:
            pro_grp = pro_yr.groupby(['monitor_month', 'month'])['yhat'].sum().unstack('month')
            pro_grp.columns = [int(c) for c in pro_grp.columns]

    monitors_in_year = sorted(df_yr['monitor_month'].unique())
    q_monitors_yr = [m for m in monitors_in_year if int(m.split('-')[1]) in [1, 4, 7, 10]]
    months = list(range(1, 13))
    pivot_rows = []

    for mon in q_monitors_yr:
        row_0 = sql_grp.loc[(mon, 0)].to_dict() if (mon, 0) in sql_grp.index else {}
        row_1 = sql_grp.loc[(mon, 1)].to_dict() if (mon, 1) in sql_grp.index else {}
        row_p = pro_grp.loc[mon].to_dict() if mon in pro_grp.index else {}

        if pd.notna(row_0.get(12, np.nan)):
            row_1 = {}; row_p = {}

        if row_1: row_p = {m: v for m, v in row_p.items() if pd.notna(row_1.get(m, np.nan))}
        else: row_p = {}

        mon_data = {}
        for m in months:
            if pd.notna(row_0.get(m, np.nan)): mon_data[m] = row_0[m]
            elif pd.notna(row_1.get(m, np.nan)): mon_data[m] = row_1[m]

        def make_row(label, data):
            r = {'Spesialisasi': target_spec, 'Monitor_Month': mon, 'Tipe': label}
            grand = 0
            for m in months:
                v = data.get(m, np.nan)
                r[str(m)] = round(v) if pd.notna(v) else np.nan
                if pd.notna(v): grand += v
            r['Grand Total'] = round(grand)
            return r

        pivot_rows.append(make_row(f'■ {mon}', mon_data))
        if row_0: pivot_rows.append(make_row('  0  Aktual', row_0))
        if row_1: pivot_rows.append(make_row('  1  Forecast SQL', row_1))
        if row_p: pivot_rows.append(make_row('  P  Forecast Prophet', row_p))

    if not pivot_rows: return pd.DataFrame()
    
    gt_data = {}
    for m in months:
        gt_data[m] = sql_grp[m].sum() if m in sql_grp.columns else np.nan
    pivot_rows.append(make_row('GRAND TOTAL', gt_data))
    
    return pd.DataFrame(pivot_rows)

# Pipeline Data Split & Ekspor Detail

In [24]:
def compute_insurer_split(df_raw: pd.DataFrame, min_volume: int = 10) -> pd.DataFrame:
    """
    Menghitung proporsi market share asuransi (payer) berdasarkan level hierarki: 
    Product, Specialism, atau Hospital Total.
    """
    insurer_col = None
    for cand in ['uzovi_code', 'payer_code']:
        if cand in df_raw.columns:
            insurer_col = cand
            break
            
    if insurer_col is None:
        print("⚠️  Kolom uzovi_code / payer_code tidak ditemukan.")
        return pd.DataFrame()
        
    df = df_raw.copy()
    if 'is_forecast' in df.columns:
        df = df[df['is_forecast'] == 0]
        
    vol_col = 'volume' if 'volume' in df.columns else 'y'
    if df[vol_col].dtype in ['object', 'O']:
        df[vol_col] = df[vol_col].astype(str).str.replace(',', '.').astype(float)
        
    prod_total_map = df.groupby('product_code')[vol_col].sum().rename('prod_total')
    prod_ins_share = df.groupby(['product_code', insurer_col])[vol_col].sum().reset_index().rename(columns={vol_col: 'ins_vol'})
    prod_ins_share = prod_ins_share.merge(prod_total_map.reset_index(), on='product_code')
    prod_ins_share['market_share'] = prod_ins_share['ins_vol'] / prod_ins_share['prod_total']
    
    spec_total_map = df.groupby('specialism_code')[vol_col].sum().rename('spec_total')
    spec_ins_share = df.groupby(['specialism_code', insurer_col])[vol_col].sum().reset_index().rename(columns={vol_col: 'ins_vol'})
    spec_ins_share = spec_ins_share.merge(spec_total_map.reset_index(), on='specialism_code')
    spec_ins_share['market_share'] = spec_ins_share['ins_vol'] / spec_ins_share['spec_total']
    
    hosp_total = df[vol_col].sum()
    hosp_ins_share = df.groupby(insurer_col)[vol_col].sum().reset_index().rename(columns={vol_col: 'ins_vol'})
    hosp_ins_share['market_share'] = hosp_ins_share['ins_vol'] / hosp_total
    hosp_ins_share['split_level'] = 3
    
    all_combos = df.dropna(subset=['product_code', 'specialism_code'])[['product_code', 'specialism_code']].drop_duplicates()
    result_rows = []
    
    for _, row in all_combos.iterrows():
        prod = row['product_code']
        spec = row['specialism_code']
        prod_total = prod_total_map.get(prod, 0)
        spec_total = spec_total_map.get(spec, 0)
        
        if prod_total >= min_volume:
            lvl_rows = prod_ins_share[prod_ins_share['product_code'] == prod]
            for _, r in lvl_rows.iterrows():
                result_rows.append({'product_code': prod, 'specialism_code': spec, 'uzovi_code': r[insurer_col], 'market_share': r['market_share'], 'split_level': 1})
        elif spec_total >= min_volume:
            lvl_rows = spec_ins_share[spec_ins_share['specialism_code'] == spec]
            for _, r in lvl_rows.iterrows():
                result_rows.append({'product_code': prod, 'specialism_code': spec, 'uzovi_code': r[insurer_col], 'market_share': r['market_share'], 'split_level': 2})
        else:
            for _, r in hosp_ins_share.iterrows():
                result_rows.append({'product_code': prod, 'specialism_code': spec, 'uzovi_code': r[insurer_col], 'market_share': r['market_share'], 'split_level': 3})
                
    split_df = pd.DataFrame(result_rows)
    return split_df


def build_granular_lookup(df_raw: pd.DataFrame) -> pd.DataFrame:
    """Membangun matriks pembobot relasional spesialisasi menuju dimensi rinci (diagnosis, tipe perawatan, dst)."""
    df = df_raw.copy()
    if 'is_forecast' in df.columns:
        df = df[df['is_forecast'] == 0]
        
    vol_col = 'volume' if 'volume' in df.columns else 'y'
    dim_cols = [c for c in ['specialism_code', 'product_code', 'diagnosis_code', 'care_type_code', 'care_type'] if c in df.columns]
    
    if vol_col in df.columns and df[vol_col].dtype in ['object', 'O']:
        df[vol_col] = df[vol_col].astype(str).str.replace(',', '.').astype(float)
        
    grp = df.groupby(dim_cols, dropna=False)[vol_col].sum().reset_index().rename(columns={vol_col: 'combo_vol'})
    spec_total = grp.groupby('specialism_code')['combo_vol'].sum().rename('spec_vol')
    grp = grp.merge(spec_total, on='specialism_code')
    grp['combo_weight'] = grp['combo_vol'] / grp['spec_vol'].replace(0, np.nan)
    grp['combo_weight'] = grp['combo_weight'].fillna(0)
    w_sum = grp.groupby('specialism_code')['combo_weight'].transform('sum')
    grp['combo_weight'] = grp['combo_weight'] / w_sum.replace(0, 1)
    
    return grp


def build_forecast_detail(df_raw: pd.DataFrame, df_prophet_master: pd.DataFrame, split_df: pd.DataFrame, monitor_month: str = None) -> pd.DataFrame:
    """Menguraikan Prediksi Prophet Makro menjadi granularisasi setara dengan format output dari SQL pipeline."""
    if df_prophet_master.empty or split_df.empty or 'uzovi_code' not in split_df.columns:
        return pd.DataFrame()
        
    rev_per_vol = {}
    if 'revenue' in df_raw.columns and 'volume' in df_raw.columns:
        df_act = df_raw.copy()
        if 'is_forecast' in df_act.columns: df_act = df_act[df_act['is_forecast'] == 0]
        if df_act['volume'].dtype in ['object', 'O']: df_act['volume'] = df_act['volume'].astype(str).str.replace(',', '.').astype(float)
        grp_rev = df_act.groupby('specialism_code').apply(lambda x: (x['revenue'].sum() / x['volume'].sum() if x['volume'].sum() > 0 else 0))
        rev_per_vol = grp_rev.to_dict()
        
    granular_df = build_granular_lookup(df_raw)
    all_monitors = sorted(df_prophet_master['monitor_month'].unique())
    q_monitors_avail = [m for m in all_monitors if int(str(m).split('-')[1]) in [1, 4, 7, 10]]
    
    fc_all = df_prophet_master.copy()
    fc_all['ds'] = pd.to_datetime(fc_all['ds'])
    fc_q = fc_all[fc_all['monitor_month'].isin(q_monitors_avail)].copy()
    if fc_q.empty: return pd.DataFrame()
    
    fc_q['monitor_month_dt'] = pd.to_datetime(fc_q['monitor_month'])
    
    # --- BAGIAN YANG DIPERBAIKI (Pandas 3.0 Fix) ---
    def pick_best(group):
        ds_val = group.name[1]
        valid = group[group['monitor_month_dt'] <= ds_val]
        if valid.empty: valid = group
        res = group.loc[[valid['monitor_month_dt'].idxmax()]].copy()
        # Masukkan kembali kolom yang di-drop oleh groupby apply pandas 3.0
        res['specialism_code'] = group.name[0]
        res['ds'] = group.name[1]
        return res
    # -----------------------------------------------
        
    fc_best = fc_q.groupby(['specialism_code', 'ds'], group_keys=False).apply(pick_best).reset_index(drop=True).rename(columns={'yhat': 'vol_spec_total'})
    fc_best['vol_spec_total'] = fc_best['vol_spec_total'].clip(lower=0)
    
    merged = fc_best.merge(granular_df, on='specialism_code', how='left')
    merged['vol_combo'] = merged['vol_spec_total'] * merged['combo_weight'].fillna(0)
    
    split_join = split_df[['product_code', 'specialism_code', 'uzovi_code', 'market_share', 'split_level']].copy()
    if 'product_code' in merged.columns:
        expanded = merged.merge(split_join, on=['product_code', 'specialism_code'], how='left')
    else:
        split_spec = split_df.groupby(['specialism_code', 'uzovi_code'])[['market_share', 'split_level']].first().reset_index()
        expanded = merged.merge(split_spec, on='specialism_code', how='left')
        
    if 'uzovi_code' in expanded.columns:
        no_uzovi = expanded['uzovi_code'].isna()
        if no_uzovi.any():
            specs_missing = expanded.loc[no_uzovi, 'specialism_code'].unique()
            spec_fallback = split_df[split_df['specialism_code'].isin(specs_missing)].groupby(['specialism_code', 'uzovi_code']).agg(market_share=('market_share', 'first'), split_level=('split_level', 'first')).reset_index()
            merged_missing = expanded[no_uzovi].drop(columns=['uzovi_code', 'market_share', 'split_level'], errors='ignore').merge(spec_fallback, on='specialism_code', how='left')
            expanded = pd.concat([expanded[~no_uzovi], merged_missing], ignore_index=True)
            
    if expanded.empty or 'uzovi_code' not in expanded.columns: return pd.DataFrame()
    
    expanded['volume'] = (expanded['vol_combo'] * expanded['market_share'].fillna(0)).round(7)
    expanded['revenue'] = (expanded['volume'] * expanded['specialism_code'].map(rev_per_vol).fillna(0)).round(6)
    
    result = {
        'monitor_month': expanded['monitor_month'],
        'year': expanded['year'].astype(int),
        'month': expanded['month'].astype(int),
        'product_code': expanded['product_code'] if 'product_code' in expanded.columns else np.nan,
        'specialism_code': expanded['specialism_code'],
        'diagnosis_code': expanded['diagnosis_code'] if 'diagnosis_code' in expanded.columns else np.nan,
        'care_type_code': expanded['care_type_code'] if 'care_type_code' in expanded.columns else (expanded['care_type'] if 'care_type' in expanded.columns else np.nan),
        'payer_code': expanded['uzovi_code'],
        'split_level': expanded['split_level'] if 'split_level' in expanded.columns else np.nan,
        'is_forecast': 1,
        'volume': expanded['volume'],
        'revenue': expanded['revenue']
    }
    
    df_out = pd.DataFrame(result)
    df_out = df_out[df_out['volume'] > 0].copy()
    return df_out.sort_values(['monitor_month', 'specialism_code', 'year', 'month']).reset_index(drop=True)


# Export Manager (CSV & Excel)

In [25]:
def export_to_csv(df_raw: pd.DataFrame, df_prophet_master: pd.DataFrame, output_path: str = "prophet_forecast_v5.csv"):
    """
    Mengekspor prediksi Prophet berdetail (granular) ke dalam CSV 
    terbatas pada periode komparasi Mei - Desember tahun target.
    """
    print(f"\n📤 Membuat export CSV: {output_path}")
    split_df = compute_insurer_split(df_raw)
    df_forecast_detail = build_forecast_detail(df_raw, df_prophet_master, split_df, None)
    
    if df_forecast_detail.empty:
        print("   ⚠️  Tidak ada data forecast sama sekali untuk diekspor CSV.")
        return
        
    df_out = df_forecast_detail[
        (df_forecast_detail['year'] == TARGET_TAHUN) &
        (df_forecast_detail['month'] >= 5) &
        (df_forecast_detail['month'] <= 12)
    ].copy()

    if df_out.empty:
        print("   ⚠️  Data CSV KOSONG. (Pastikan ada data prophet_master di rentang Mei-Des 2025)")
        return
        
    row_count_before = len(df_out)
    df_out = df_out[df_out['volume'] >= 0.0001].copy()
    row_count_after = len(df_out)
    print(f"   🧹 Cleanup: Membuang {row_count_before - row_count_after:,} baris debu (volume < 0.0001).")
    
    col_map = {
        'product_code': 'product_code', 'specialism_code': 'specialism_code', 'diagnosis_code': 'diagnosis_code',
        'care_type_code': 'care_type', 'payer_code': 'uzovi_code', 'year': 'year', 'month': 'month',
        'volume': 'volume', 'revenue': 'revenue'
    }
    
    for src_col in col_map:
        if src_col not in df_out.columns: df_out[src_col] = np.nan
        
    df_out = df_out[list(col_map.keys())].rename(columns=col_map).copy()
    df_out['volume'] = df_out['volume'].apply(lambda x: f"{float(x):.7f}".replace('.', ',') if pd.notna(x) else '')
    df_out['revenue'] = df_out['revenue'].apply(lambda x: f"{float(x):.6f}".replace('.', ',') if pd.notna(x) else '')
    df_out = df_out[['product_code', 'specialism_code', 'diagnosis_code', 'care_type', 'uzovi_code', 'year', 'month', 'volume', 'revenue']]
    
    df_out.to_csv(output_path, sep=';', index=False, encoding='utf-8-sig')
    print(f"   ✅ Export CSV selesai (Khusus Mei-Des 2025 sebagai Komparasi) → Total baris final: {len(df_out):,}")


def export_to_excel(df_raw: pd.DataFrame, df_prophet_master: pd.DataFrame, specialties_list: list, target_year: int, output_path: str = "prophet_export_v5.xlsx"):
    """
    Menyusun tabel Pivot Dataframe ke dalam file Excel bergaya korporat yang rapih.
    """
    import openpyxl
    from openpyxl.styles import PatternFill, Font, Alignment
    from openpyxl.utils import get_column_letter

    print(f"\n📤 Membuat export Excel HANYA untuk Tab Analisis Pivot: {output_path}")

    all_pivot_frames = []
    for spec in specialties_list:
        pf = build_pivot_dataframe(df_raw, df_prophet_master, target_year, spec)
        if not pf.empty:
            all_pivot_frames.append(pf)
            
    df_analisis = pd.concat(all_pivot_frames, ignore_index=True) if all_pivot_frames else pd.DataFrame()

    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        if not df_analisis.empty:
            df_export = df_analisis.drop(columns=['Spesialisasi', 'Monitor_Month'], errors='ignore').copy()
            df_export = df_export.rename(columns={'Tipe': 'Row Labels'})

            ws = writer.book.create_sheet('Analisis')
            writer.sheets['Analisis'] = ws
            ws.cell(row=1, column=1, value='year')
            ws.cell(row=1, column=2, value=target_year)
            ws.cell(row=2, column=1, value='Sum of volume Column')

            for col_idx, col_name in enumerate(df_export.columns, start=1):
                ws.cell(row=3, column=col_idx, value=col_name)

            for row_idx, data_row in enumerate(df_export.itertuples(index=False), start=4):
                for col_idx, val in enumerate(data_row, start=1):
                    ws.cell(row=row_idx, column=col_idx, value=val)

            # Styling Blok Excel
            header_fill = PatternFill(fill_type='solid', fgColor='1F4E79')
            header_font = Font(color='FFFFFF', bold=True, size=10)
            fill_a      = PatternFill(fill_type='solid', fgColor='DCE6F1')
            fill_b      = PatternFill(fill_type='solid', fgColor='FFFFFF')
            grand_fill  = PatternFill(fill_type='solid', fgColor='BDD7EE')

            ws.cell(row=1, column=1).font = Font(bold=True, size=10)
            ws.cell(row=1, column=2).font = Font(bold=True, size=10)
            ws.cell(row=2, column=1).font = Font(bold=True, size=10)

            n_cols = len(df_export.columns)
            for col_idx in range(1, n_cols + 1):
                cell = ws.cell(row=3, column=col_idx)
                cell.fill = header_fill
                cell.font = header_font
                cell.alignment = Alignment(horizontal='center', vertical='center')

            for row_idx in range(4, ws.max_row + 1):
                label_val  = str(ws.cell(row=row_idx, column=1).value or '')
                is_grand   = 'GRAND TOTAL' in label_val
                is_summary = label_val.startswith('■')
                fill = grand_fill if is_grand else (fill_a if row_idx % 2 == 0 else fill_b)

                for col_idx in range(1, n_cols + 1):
                    cell = ws.cell(row=row_idx, column=col_idx)
                    cell.fill = fill
                    if is_grand or is_summary: cell.font = Font(bold=True, size=9)
                    else: cell.font = Font(size=9)
                    if col_idx > 1:
                        cell.number_format = '#,##0'
                        cell.alignment = Alignment(horizontal='right')

            ws.column_dimensions['A'].width = 28
            for col_idx in range(2, n_cols + 1):
                ws.column_dimensions[get_column_letter(col_idx)].width = 10
                
            ws.freeze_panes = 'A4'
            print(f"   ✅ Tab 'Analisis'      → {len(df_export)} baris, {len(df_export.columns)} kolom")
        else:
            print("   ⚠️  Tidak ada data untuk Analisis Pivot.")

    print(f"\n✅ Export Excel selesai (Tanpa Forecast Data Tab) → {output_path}\n")

# Eksekusi Pivot Terminal & Eksport Data Final

In [ ]:
# Ekstrak Master Spesialisasi dari data bersih
specialties_list = df_clean['specialism_code'].unique().tolist()

# 1. Print Text Pivot di Terminal per Spesialisasi
for spec in specialties_list:
    print_text_pivot(df_raw, df_prophet_master, TARGET_TAHUN, spec)

# 2. Export ke CSV (Raw Forecast detail, membuang debu, murni untuk bulan 5-12 2025)
export_to_csv(
    df_raw=df_raw,
    df_prophet_master=df_prophet_master,
    output_path="data/output/prophet_forecast.csv"
)

# 3. Export ke Excel (Hanya sheet Pivot Analisis)
export_to_excel(
    df_raw=df_raw,
    df_prophet_master=df_prophet_master,
    specialties_list=specialties_list,
    target_year=TARGET_TAHUN,
    output_path="data/output/prophet_pivot_table.xlsx"
)


  PIVOT TABLE — Spesialisasi: AGB:0303  |  Filter: Tahun = 2025
Monitor / Tipe                        1        2        3        4        5        6        7        8        9       10       11       12  Grand Total
--------------------------------------------------------------------------------------------------------------------------------------------
■ 2025-04-01                      3.250    2.906    3.416    3.456    2.978    2.967    3.177    3.161    3.102    3.311    3.033    2.952       37.710
    0  Aktual                     3.250    2.906    3.416    3.456                                                                               13.028
    1  Forecast SQL                                                   2.978    2.967    3.177    3.161    3.102    3.311    3.033    2.952       24.682
    P  Forecast Prophet                                               3.084    3.093    3.384    3.168    3.123    3.432    3.111    3.068       25.464
----------------------------------